# Dataset

> Base class of episode datasets: clips of `num_steps` steps for training (stable-worldmodel API), and the datasets composing them: `MergeDataset` (columns of several datasets), `ConcatDataset` (episodes of several datasets) and `GoalDataset` (a sampled goal per item).

In [ ]:
#| default_exp data.dataset

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Adapted from stable-worldmodel's data package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import os
from pathlib import Path
from typing import Callable

import numpy as np

In [ ]:
#| export
def get_cache_dir(override_root: str | Path | None = None, sub_folder: str | None = None) -> Path:
    "Where named datasets and checkpoints live: `override_root`, else ``$STABLEMARL_HOME``, else ``~/.stable_marl``."
    root = Path(override_root or os.environ.get('STABLEMARL_HOME', '~/.stable_marl')).expanduser()
    path = root / sub_folder if sub_folder else root
    path.mkdir(parents=True, exist_ok=True)
    return path


def to_tensor(data: np.ndarray):
    """
    A column slice as a torch tensor. As in stable-worldmodel, images of shape (T, H, W, C)
    with 1 or 3 channels become channel-first, (T, C, H, W); per-agent images
    (T, num_agents, H, W, C) are left as they are.
    """
    import torch
    tensor = torch.from_numpy(np.ascontiguousarray(data))
    if tensor.ndim == 4 and tensor.shape[-1] in (1, 3):
        tensor = tensor.permute(0, 3, 1, 2)
    return tensor

In [ ]:
#| export
class Dataset:
    """
    Base class of episode datasets, stored as flat columns with episode ``lengths`` / ``offsets``.

    Item ``i`` is a clip of ``num_steps`` steps, ``frameskip`` apart: a dict of torch tensors,
    one per column. The ``action`` column keeps every step of the clip and is reshaped to
    ``(num_steps, -1)``, so with ``frameskip=1`` it is ``(num_steps, num_agents)``.
    Subclasses implement ``column_names`` and ``_load_slice``.
    """
    def __init__(
        self,
        lengths: np.ndarray,
        offsets: np.ndarray,
        frameskip: int = 1,
        num_steps: int = 1,
        transform: Callable[[dict], dict] | None = None):
        self.lengths, self.offsets = lengths, offsets
        self.frameskip, self.num_steps = frameskip, num_steps
        self.span = num_steps * frameskip
        self.transform = transform
        self.clip_indices = [(ep, start) for ep, length in enumerate(lengths) if length >= self.span
                             for start in range(length - self.span + 1)]

    @property
    def column_names(self) -> list[str]:
        raise NotImplementedError

    @property
    def episode_column_names(self) -> list[str]:
        "Columns with one value per episode (not per step): none by default."
        return []

    def get_episode_data(self, episodes_idx: np.ndarray | list[int] | None = None) -> dict[str, list]:
        "The episode-scoped values ``{name: [value per requested episode]}`` (None: every episode)."
        return {}

    def _load_slice(self, ep_idx: int, start: int, end: int) -> dict:
        raise NotImplementedError

    def __len__(self) -> int:
        return len(self.clip_indices)

    def __getitem__(self, idx: int) -> dict:
        ep_idx, start = self.clip_indices[idx]
        steps = self._load_slice(ep_idx, start, start + self.span)
        if 'action' in steps:
            steps['action'] = steps['action'].reshape(self.num_steps, -1)
        return steps

    def load_chunk(self, episodes_idx: np.ndarray, start: np.ndarray, end: np.ndarray) -> list[dict]:
        "One slice ``[start, end)`` per episode, in order."
        chunk = []
        for ep, s, e in zip(episodes_idx, start, end):
            steps = self._load_slice(ep, s, e)
            if 'action' in steps:
                steps['action'] = steps['action'].reshape((e - s) // self.frameskip, -1)
            chunk.append(steps)
        return chunk

    def load_episode(self, episode_idx: int) -> dict:
        "Every step of one episode."
        return self._load_slice(episode_idx, 0, self.lengths[episode_idx])

    def get_col_data(self, col: str) -> np.ndarray:
        "The whole column, rows of every episode one after the other."
        raise NotImplementedError

    def get_dim(self, col: str) -> int:
        "Number of values per row of `col`."
        data = self.get_col_data(col)
        return int(np.prod(data.shape[1:])) if data.ndim > 1 else 1

    def get_row_data(self, row_idx: int | list[int]) -> dict:
        "Every column at the given flat storage row(s)."
        raise NotImplementedError

    def merge_col(self, source: list[str] | str, target: str, dim: int = -1) -> None:
        "Concatenate the `source` column(s) along `dim` into a new column `target`."
        raise NotImplementedError

## Composing datasets

As in stable-worldmodel: `MergeDataset` joins the columns of datasets with the same clips (e.g. the
same file read with different `keys_to_load`), `ConcatDataset` stacks the episodes of several
datasets, and `GoalDataset` adds a goal observation sampled per item (from a later step of the same
episode, a random step, or the current one). Columns are kept as they are, per-agent axis included.

In [ ]:
#| export
def _first_seen(names) -> list[str]:
    seen, out = set(), []
    for n in names:
        if n not in seen:
            seen.add(n); out.append(n)
    return out


class MergeDataset:
    """
    Columns of several datasets with the same clips (horizontal join).

    `keys_from_dataset` gives the columns taken from each dataset; by default each dataset
    contributes the columns not already given by the earlier ones.
    """
    def __init__(self, datasets: list, keys_from_dataset: list[list[str]] | None = None):
        if not datasets:
            raise ValueError("Need at least one dataset")
        self.datasets, self._len = datasets, len(datasets[0])
        if keys_from_dataset:
            self.keys_map = keys_from_dataset
        else:
            seen, self.keys_map = set(), []
            for ds in datasets:
                keys = [c for c in ds.column_names if c not in seen]
                seen.update(keys)
                self.keys_map.append(keys)

    @property
    def column_names(self) -> list[str]:
        return [c for keys in self.keys_map for c in keys]

    @property
    def episode_column_names(self) -> list[str]:
        return _first_seen(c for ds in self.datasets for c in getattr(ds, 'episode_column_names', []))

    def get_episode_data(self, episodes_idx=None) -> dict[str, list]:
        "Episode data of every dataset (the first one wins on a name collision)."
        out = {}
        for ds in self.datasets:
            getter = getattr(ds, 'get_episode_data', None)
            for k, v in (getter(episodes_idx) if getter else {}).items():
                out.setdefault(k, v)
        return out

    @property
    def lengths(self) -> np.ndarray:
        return self.datasets[0].lengths

    def __len__(self) -> int:
        return self._len

    def __getitem__(self, idx: int) -> dict:
        out = {}
        for ds, keys in zip(self.datasets, self.keys_map):
            item = ds[idx]
            out.update({k: item[k] for k in keys if k in item})
        return out

    def load_chunk(self, episodes_idx: np.ndarray, start: np.ndarray, end: np.ndarray) -> list[dict]:
        merged = []
        for items in zip(*(ds.load_chunk(episodes_idx, start, end) for ds in self.datasets)):
            combined = {}
            for item in items:
                combined.update(item)
            merged.append(combined)
        return merged

    def get_col_data(self, col: str) -> np.ndarray:
        for ds, keys in zip(self.datasets, self.keys_map):
            if col in keys:
                return ds.get_col_data(col)
        raise KeyError(col)

    def get_row_data(self, row_idx: int | list[int]) -> dict:
        out = {}
        for ds, keys in zip(self.datasets, self.keys_map):
            data = ds.get_row_data(row_idx)
            out.update({k: data[k] for k in keys if k in data})
        return out


class ConcatDataset:
    """
    Episodes of several datasets one after the other (vertical join): the clip and episode
    indices of a dataset come after those of the datasets before it.
    """
    def __init__(self, datasets: list):
        if not datasets:
            raise ValueError("Need at least one dataset")
        self.datasets = datasets
        self._cum = np.cumsum([0] + [len(ds) for ds in datasets])
        self._ep_cum = np.cumsum([0] + [len(ds.lengths) for ds in datasets])

    @property
    def column_names(self) -> list[str]:
        return _first_seen(c for ds in self.datasets for c in ds.column_names)

    @property
    def episode_column_names(self) -> list[str]:
        return _first_seen(c for ds in self.datasets for c in getattr(ds, 'episode_column_names', []))

    @property
    def lengths(self) -> np.ndarray:
        return np.concatenate([np.asarray(ds.lengths) for ds in self.datasets])

    def _split_episodes(self, episodes_idx) -> tuple[np.ndarray, np.ndarray]:
        "(dataset of each global episode, its local index there)"
        episodes_idx = np.asarray(episodes_idx, dtype=np.int64).reshape(-1)
        ds_indices = np.searchsorted(self._ep_cum[1:], episodes_idx, side='right')
        return ds_indices, episodes_idx - self._ep_cum[ds_indices]

    def get_episode_data(self, episodes_idx=None) -> dict[str, list]:
        "Episode data through the global -> local episode index (None for datasets missing a key)."
        cols = self.episode_column_names
        if not cols:
            return {}
        if episodes_idx is None:
            episodes_idx = np.arange(int(self._ep_cum[-1]))
        ds_indices, local_eps = self._split_episodes(episodes_idx)
        out = {k: [None] * len(ds_indices) for k in cols}
        for d, ds in enumerate(self.datasets):
            mask = ds_indices == d
            getter = getattr(ds, 'get_episode_data', None)
            if not mask.any() or getter is None:
                continue
            for k, vals in getter(local_eps[mask]).items():
                for pos, v in zip(np.flatnonzero(mask), vals):
                    out[k][int(pos)] = v
        return out

    def __len__(self) -> int:
        return int(self._cum[-1])

    def _loc(self, idx: int) -> tuple[int, int]:
        if idx < 0:
            idx += len(self)
        ds_idx = int(np.searchsorted(self._cum[1:], idx, side='right'))
        return ds_idx, int(idx - self._cum[ds_idx])

    def __getitem__(self, idx: int) -> dict:
        ds_idx, local_idx = self._loc(idx)
        return self.datasets[ds_idx][local_idx]

    def __getitems__(self, indices: list[int]) -> list[dict]:
        "Batched access (used by DataLoader): each dataset is read once, with its own ``__getitems__`` if it has one."
        groups: dict[int, list[tuple[int, int]]] = {}
        for pos, (ds_idx, local_idx) in enumerate(self._loc(i) for i in indices):
            groups.setdefault(ds_idx, []).append((pos, local_idx))
        results: list = [None] * len(indices)
        for ds_idx, items in groups.items():
            ds, local = self.datasets[ds_idx], [i for _, i in items]
            fetched = ds.__getitems__(local) if hasattr(ds, '__getitems__') else [ds[i] for i in local]
            for (pos, _), item in zip(items, fetched):
                results[pos] = item
        return results

    def load_chunk(self, episodes_idx: np.ndarray, start: np.ndarray, end: np.ndarray) -> list[dict]:
        "One slice per (global) episode, each read from its dataset."
        start, end = np.asarray(start), np.asarray(end)
        ds_indices, local_eps = self._split_episodes(episodes_idx)
        results: list = [None] * len(ds_indices)
        for d, ds in enumerate(self.datasets):
            mask = ds_indices == d
            if mask.any():
                for i, chunk in zip(np.flatnonzero(mask), ds.load_chunk(local_eps[mask], start[mask], end[mask])):
                    results[i] = chunk
        return results

    def get_col_data(self, col: str) -> np.ndarray:
        data = [ds.get_col_data(col) for ds in self.datasets if col in ds.column_names]
        if not data:
            raise KeyError(col)
        return np.concatenate(data)

    def get_row_data(self, row_idx: int | list[int]) -> dict:
        "The given global row(s) (rows of the datasets one after the other), stacked for a list."
        rows = np.cumsum([0] + [int(np.sum(ds.lengths)) for ds in self.datasets])
        def one(r):
            d = int(np.searchsorted(rows[1:], r, side='right'))
            return self.datasets[d].get_row_data(int(r - rows[d]))
        if isinstance(row_idx, (int, np.integer)):
            return one(int(row_idx))
        results: dict[str, list] = {}
        for r in row_idx:
            for k, v in one(int(r)).items():
                results.setdefault(k, []).append(v)
        return {k: np.stack(v) for k, v in results.items()}

In [ ]:
#| export
GOAL_KINDS = ('random', 'geometric_future', 'uniform_future', 'current')


class GoalDataset:
    """
    Wraps a dataset to add a goal, sampled per item, under ``goal_<key>`` for each `goal_keys` column.

    The goal step is drawn (probabilities `goal_probabilities`, in the order of :data:`GOAL_KINDS`):

    * ``'random'``: any step of the dataset
    * ``'geometric_future'``: a later step of the same episode, ``Geom(1 - gamma)`` steps
      (of ``frameskip``) after the clip's current step
    * ``'uniform_future'``: a later step of the same episode, uniformly
    * ``'current'``: the clip's current step (``current_goal_offset`` steps into the clip,
      by default its last)

    Clips without a later step are left out when future goals can be drawn.

    Parameters
    ----------
    goal_keys : dict
        Source column -> goal column. Default: ``pixels`` (each agent's view),
        ``proprio`` and ``position``, when in the dataset, to ``goal_<key>``
    seed : int, optional
        Seed of the goal sampling
    """
    def __init__(
        self,
        dataset: Dataset,
        goal_probabilities: tuple[float, float, float, float] = (0.3, 0.5, 0.0, 0.2),
        gamma: float = 0.99,
        current_goal_offset: int | None = None,
        goal_keys: dict[str, str] | None = None,
        seed: int | None = None):
        if len(goal_probabilities) != 4:
            raise ValueError(f"goal_probabilities must be a 4-tuple {GOAL_KINDS}")
        if not np.isclose(sum(goal_probabilities), 1.0):
            raise ValueError("goal_probabilities must sum to 1.0")
        self.dataset, self.goal_probabilities, self.gamma = dataset, goal_probabilities, gamma
        self.current_goal_offset = dataset.num_steps if current_goal_offset is None else current_goal_offset
        self.rng = np.random.default_rng(seed)
        self.episode_lengths, self.episode_offsets = dataset.lengths, dataset.offsets
        self._episode_cumlen = np.cumsum(self.episode_lengths)
        self._total_steps = int(self._episode_cumlen[-1]) if len(self._episode_cumlen) else 0
        if goal_keys is None:
            goal_keys = {k: f'goal_{k}' for k in ('pixels', 'proprio', 'position') if k in dataset.column_names}
        self.goal_keys = goal_keys

        if goal_probabilities[1] > 0 or goal_probabilities[2] > 0:   # keep clips with a future step
            current_end_offset = (self.current_goal_offset - 1) * dataset.frameskip
            kept = [(i, (ep, start)) for i, (ep, start) in enumerate(dataset.clip_indices)
                    if start + current_end_offset + dataset.frameskip < self.episode_lengths[ep]]
            self._index_mapping = [i for i, _ in kept]
            self._clip_indices = [clip for _, clip in kept]
        else:
            self._clip_indices = list(dataset.clip_indices)
            self._index_mapping = list(range(len(dataset.clip_indices)))

    @property
    def clip_indices(self) -> list[tuple[int, int]]:
        return self._clip_indices

    def __len__(self) -> int:
        return len(self._clip_indices)

    @property
    def column_names(self) -> list[str]:
        "Columns of the wrapped dataset (the goal columns are added per item)."
        return self.dataset.column_names

    @property
    def episode_column_names(self) -> list[str]:
        return getattr(self.dataset, 'episode_column_names', [])

    def get_episode_data(self, episodes_idx=None) -> dict[str, list]:
        getter = getattr(self.dataset, 'get_episode_data', None)
        return getter(episodes_idx) if getter is not None else {}

    def _sample_goal_kind(self) -> str:
        r, edges = self.rng.random(), np.cumsum(self.goal_probabilities)
        for kind, edge in zip(GOAL_KINDS[:3], edges[:3]):
            if r < edge:
                return kind
        return 'current'

    def _sample_random_step(self) -> tuple[int, int]:
        if self._total_steps == 0:
            return 0, 0
        flat_idx = int(self.rng.integers(0, self._total_steps))
        ep_idx = int(np.searchsorted(self._episode_cumlen, flat_idx, side='right'))
        return ep_idx, int(flat_idx - (self._episode_cumlen[ep_idx - 1] if ep_idx > 0 else 0))

    def _future_steps(self, ep_idx: int, local_start: int) -> tuple[int, int]:
        "(current step, number of later steps available, in frameskips)"
        frameskip = self.dataset.frameskip
        current_end = local_start + (self.current_goal_offset - 1) * frameskip
        max_steps = (self.episode_lengths[ep_idx] - 1 - current_end) // frameskip
        assert max_steps >= 1, f"No future frames available: {max_steps=}"
        return current_end, max_steps

    def _sample_geometric_future_step(self, ep_idx: int, local_start: int) -> tuple[int, int]:
        current_end, max_steps = self._future_steps(ep_idx, local_start)
        k = min(int(self.rng.geometric(max(1.0 - self.gamma, 1e-6))), max_steps)
        return ep_idx, current_end + k * self.dataset.frameskip

    def _sample_uniform_future_step(self, ep_idx: int, local_start: int) -> tuple[int, int]:
        current_end, max_steps = self._future_steps(ep_idx, local_start)
        return ep_idx, current_end + int(self.rng.integers(1, max_steps + 1)) * self.dataset.frameskip

    def _get_clip_info(self, idx: int) -> tuple[int, int]:
        "``(episode, start)`` of clip `idx`."
        return self._clip_indices[idx]

    def _load_single_step(self, ep_idx: int, local_idx: int) -> dict:
        return self.dataset._load_slice(ep_idx, local_idx, local_idx + 1)

    def __getitem__(self, idx: int) -> dict:
        steps = self.dataset[self._index_mapping[idx]]
        if not self.goal_keys:
            return steps
        ep_idx, local_start = self._get_clip_info(idx)
        kind = self._sample_goal_kind()
        if kind == 'random':
            goal_ep, goal_idx = self._sample_random_step()
        elif kind == 'geometric_future':
            goal_ep, goal_idx = self._sample_geometric_future_step(ep_idx, local_start)
        elif kind == 'uniform_future':
            goal_ep, goal_idx = self._sample_uniform_future_step(ep_idx, local_start)
        else:
            goal_ep, goal_idx = ep_idx, local_start + (self.current_goal_offset - 1) * self.dataset.frameskip
        goal_step = self._load_single_step(goal_ep, goal_idx)
        for src_key, goal_key in self.goal_keys.items():
            if src_key in goal_step and src_key in steps:
                value = goal_step[src_key]
                steps[goal_key] = value.unsqueeze(0) if value.ndim == 0 else value
        return steps

### Tests

In [ ]:
import tempfile, torch
from fastcore.test import test_fail
from stable_marl.data import HDF5Dataset, HDF5Writer

def episode(T, ep, A=2):
    "Episode `ep` of length T: every column tells the episode and the step."
    return {'pixels': np.full((T, A, 4, 4, 3), ep, np.uint8), 'position': np.stack([np.stack([np.arange(T), np.full(T, ep)], 1)] * A, 1),
            'action': np.full((T, A), ep, np.float32), 'step_idx': np.arange(T), 'ep': np.full(T, ep)}

with tempfile.TemporaryDirectory() as tmp:
    for name, eps in (('a', [(5, 0), (3, 1)]), ('b', [(4, 2)])):
        with HDF5Writer(f'{tmp}/{name}.h5') as w:
            w.write_episodes(episode(T, ep) for T, ep in eps)
    a, b = HDF5Dataset(path=f'{tmp}/a.h5', num_steps=2), HDF5Dataset(path=f'{tmp}/b.h5', num_steps=2)
    assert a.episode_column_names == [] and a.get_episode_data() == {}

    # MergeDataset: columns of the same clips from two readers
    m = MergeDataset([HDF5Dataset(path=f'{tmp}/a.h5', num_steps=2, keys_to_load=['pixels', 'ep']),
                      HDF5Dataset(path=f'{tmp}/a.h5', num_steps=2, keys_to_load=['action', 'ep', 'step_idx'])])
    assert m.column_names == ['pixels', 'ep', 'action', 'step_idx'] and len(m) == len(a) == 6
    item = m[5]
    assert set(item) == {'pixels', 'ep', 'action', 'step_idx'} and item['ep'].tolist() == [1, 1]
    assert item['pixels'].shape == (2, 2, 4, 4, 3) and item['action'].shape == (2, 2)
    assert m.load_chunk(np.array([1]), np.array([0]), np.array([3]))[0]['step_idx'].tolist() == [0, 1, 2]
    assert m.get_col_data('action').shape == (8, 2) and m.get_row_data(5)['ep'] == 1
    test_fail(lambda: m.get_col_data('nope'), contains='nope')

    # ConcatDataset: the episodes of b after those of a
    c = ConcatDataset([a, b])
    assert len(c) == len(a) + len(b) == 9 and c.lengths.tolist() == [5, 3, 4]
    assert c[len(a)]['ep'].tolist() == [2, 2] and c[-1]['step_idx'].tolist() == [2, 3]
    assert [x['ep'][0].item() for x in c.__getitems__([0, 8, 4])] == [0, 2, 1]
    chunk = c.load_chunk(np.array([2, 0]), np.array([1, 0]), np.array([3, 2]))
    assert chunk[0]['ep'].tolist() == [2, 2] and chunk[1]['ep'].tolist() == [0, 0]
    assert c.get_col_data('ep').tolist() == [0] * 5 + [1] * 3 + [2] * 4
    assert c.get_row_data(8)['ep'] == 2 and c.get_row_data([0, 9])['ep'].tolist() == [0, 2]

    # GoalDataset: future goals come later in the same episode; each agent's view and position
    g = GoalDataset(a, goal_probabilities=(0.0, 0.5, 0.5, 0.0), seed=0)
    assert len(g) == 4                                  # clips ending on an episode's last step have no future
    for i in range(len(g)):
        for _ in range(5):
            item = g[i]
            assert item['goal_pixels'].shape == (1, 2, 4, 4, 3) and item['goal_position'].shape == (1, 2, 2)
            assert (item['goal_pixels'] == item['ep'][0]).all()                         # same episode
            assert (item['goal_position'][0, :, 0] > item['step_idx'][-1]).all()        # a later step
    current = GoalDataset(a, goal_probabilities=(0.0, 0.0, 0.0, 1.0))
    assert len(current) == len(a)
    item = current[2]
    assert torch.equal(item['goal_position'][0], item['position'][-1])                  # the clip's last step
    rand = GoalDataset(a, goal_probabilities=(1.0, 0.0, 0.0, 0.0), goal_keys={'ep': 'goal_ep'}, seed=1)
    assert {int(rand[0]['goal_ep'][0]) for _ in range(30)} == {0, 1}                    # any episode
    test_fail(lambda: GoalDataset(a, goal_probabilities=(0.5, 0.5, 0.5, 0.0)), contains='sum to 1')
    for ds in (a, b): ds.close()
    for ds in m.datasets: ds.close()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()